# CISA Known Exploited Vulnerabilities (KEV) Neural Networks
Data source: The Cybersecurity and Infrastructure Security Agency's (CISA) [Known Exploited Vulnerabilities (KEV) catalog](https://github.com/cisagov/kev-data)

## Background
**Exploratory Analysis**
<br>
I previously performed an exploratory data analysis on the KEV dataset to identify trends in known exploited vulnerabilities, remediation-timelines, and ransomware-associated weaknesses. Find my exploratory analysis [here](https://github.com/ClarenceBallensky/CISA_KEV_Analysis).

My findings were as follows:
- Microsoft has the largest number of cataloged vulnerabilities.
- Input validation and use-after-free are the most common weaknesses.
- Ransomware vulnerabilities show similar remediation timelines to the broader catalog.
- Most remediation deadlines are exactly 21 days.
- Vulnerability additions peaked in 2022.

**Machine Learning**
<br>
I also built several machine learning models to classify ransomware status--including a logistic regression, random forest, and support vector machine model. I then compared performance. 
Find my machine learning models [here](https://github.com/ClarenceBallensky/CISA_KEV_Machine-Learning).

My models ranked as follows:
1) Logistic regression
2) Random forest
3) Support vector machine

My best model, logistic regression, produced the following scores:
<br>
Accuracy score: 0.8
<br>
Precision score: 0.5016611295681063
<br>
Recall score: 0.45481927710843373
<br>
f1 score: 0.47709320695102686
<br>

Note that accuracy alone is not a meaningful benchmark here, since a model that always predicts 'Unknown' would already score ~80%. The logistic regression model's real gains were in precision and recall--catching a meaningfully higher share of actual ransomware-associated CVEs than a naive baseline would.

## Step 1: Load the Data

In [1]:
import pandas as pd

# Get the .csv data from GitHub
data_url = "https://raw.githubusercontent.com/cisagov/kev-data/refs/heads/develop/known_exploited_vulnerabilities.csv"
kev = pd.read_csv(data_url)

# Save a local copy
kev.to_csv("known_exploited_vulnerabilities.csv", index=False)
kev = pd.read_csv("known_exploited_vulnerabilities.csv")

print(kev.head(30))

             cveID    vendorProject  \
0   CVE-2026-20349            Cisco   
1   CVE-2026-68820        Microsoft   
2   CVE-2026-72898         Metabase   
3    CVE-2026-8037         Progress   
4   CVE-2026-63077        JetBrains   
5   CVE-2026-18556           N-able   
6   CVE-2026-34486           Apache   
7    CVE-2026-9198              IBM   
8   CVE-2026-18577           N-able   
9   CVE-2026-20316            Cisco   
10  CVE-2025-68686         Fortinet   
11  CVE-2026-16812           Arista   
12  CVE-2026-16232      Check Point   
13  CVE-2026-50522        Microsoft   
14  CVE-2026-60137        WordPress   
15  CVE-2026-63030        WordPress   
16   CVE-2026-0770         Langflow   
17  CVE-2021-27137           DD-WRT   
18  CVE-2026-58644        Microsoft   
19  CVE-2026-25089         Fortinet   
20  CVE-2026-39808         Fortinet   
21  CVE-2026-46817           Oracle   
22   CVE-2023-4346  KNX Association   
23  CVE-2026-56155        Microsoft   
24  CVE-2026-56164       

## Step 2: Read the Data Documentation 
### Schema
#### (Extracted from known_exploited_vulnerabilities_schema.json)

| Column | Description |
| :--- | ---: | 
| cveID | The CVE ID of the vulnerability in the format CVE-YYYY-NNNN, note that the number portion can have more than 4 digits |
| vendorProject | The vendor or project name for the vulnerability |
| product | The vulnerability product |
| vulnerabilityName | The name of the vulnerability |
| dateAdded | The date the vulnerability was added to the catalog in the format YYYY-MM-DD |
| shortDescription | A short description of the vulnerability |
| requiredAction | The required action to address the vulnerability |
| dueDate | The date the required action is due in the format YYYY-MM-DD |
| knownRansomwareCampaignUse | 'Known' if this vulnerability is known to have been leveraged as part of a ransomware campaign; 'Unknown' if CISA lacks confirmation that the vulnerability has been utilized for ransomware |
| notes | Any additional notes about the vulnerability |
| cwes | Common Weakness Enumeration (CWE) codes associated with this vulnerability. CWEs are in the format CWE-NNNN; note that the number portion can have any number of digits |


## Step 3: Inspect the Data

In [2]:
print(kev.info())
print()
print()
print(kev.describe(include="all"))
print()
print()
kev.duplicated().sum()

<class 'pandas.DataFrame'>
RangeIndex: 1665 entries, 0 to 1664
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   cveID                       1665 non-null   str  
 1   vendorProject               1665 non-null   str  
 2   product                     1665 non-null   str  
 3   vulnerabilityName           1665 non-null   str  
 4   dateAdded                   1665 non-null   str  
 5   shortDescription            1665 non-null   str  
 6   requiredAction              1665 non-null   str  
 7   dueDate                     1665 non-null   str  
 8   knownRansomwareCampaignUse  1665 non-null   str  
 9   notes                       1665 non-null   str  
 10  cwes                        1494 non-null   str  
dtypes: str(11)
memory usage: 1.0 MB
None


                 cveID vendorProject  product  \
count             1665          1665     1665   
unique            1665           276      674  

np.int64(0)

## Step 4: Clean the Data
Before creating my models, I cleaned the dataset by renaming columns to snake_case and converting date columns to datetime objects.

In [3]:
# Rename columns to match Python's snake_case convention
kev.rename(columns={
    "cveID": "cve_id",
    "vendorProject": "vendor_project",
    "vulnerabilityName": "vulnerability_name",
    "dateAdded": "date_added",
    "shortDescription": "short_description",
    "requiredAction": "required_action",
    "dueDate": "due_date",
    "knownRansomwareCampaignUse": "known_ransomware_campaign_use"
}, inplace=True)

#print(kev.columns)



# Convert values in date_added and due_date from strings to dates
kev["date_added"] = pd.to_datetime(kev["date_added"])
kev["due_date"] = pd.to_datetime(kev["due_date"])

#print(kev.info())

## Step 5: Choosing the Model

Since I will be running this model locally, I am limited by my computer's hardware; in particular, I only have access to a CPU, not a GPU. Below is an assessment of the various models I learned about in Codecademy's "Engineer Neural Networks with PyTorch and Transformers" skill path for my use case. 

| Technique | Is Appropriate? | Explanation |
| :--- | ---- | ---: |
| DistilBERT (fine-tune) | Appropriate | Distilled BERT variant (~66M parameters); retains ~97% of BERT's language understanding at a fraction of the compute cost, making it well-suited for fine-tuning on CPU hardware |
| GRU (trained from scratch) | Has potential | Fewer gates than LSTM, reducing parameter count and training time; a reasonable sequential baseline for short-text classification |
| LSTM (trained from scratch) | Has potential | Captures longer-range dependencies via its cell state, but its additional gating mechanisms increase training time relative to GRU with limited benefit on short vulnerability descriptions |
| CIFAR10-style CNN | Inappropriate | Architecture is designed for image classification; the KEV catalog contains no image data, so there is no relevant input modality |
| CLIP fine-tuning | Inappropriate | A multimodal image-text model; excluded due to the absence of an image modality in this dataset |

<br>


**I will try fine-tuning DistilBERT and then assess its performance.**

## Step 6: Process the Data

### Importing Libraries

In [4]:
# For splitting the data
from sklearn.model_selection import train_test_split

# For converting the dataframe to a dataset
from datasets import Dataset

### Splitting the Data

In [5]:
label_map = {"Unknown": 0, "Known": 1}

text_df = pd.DataFrame({
    "text": kev["vendor_project"] + " " + kev["product"] + ": " + kev["short_description"],
    "label": kev["known_ransomware_campaign_use"].map(label_map)
})

#### Standard

The value that I am predicting, `known_ransomware_campaign_use`, is imbalanced (~20% "Known", ~80% "Unknown"). I will use include the `stratify` argument in order to preserve the ~80/20 class ratio in every fold.

In [6]:
# Splitting the data into training and testing sets

train_df, test_df = train_test_split(text_df, test_size=0.2, random_state=42, stratify=text_df["label"])

In [7]:
# Convert Pandas Dataframes to Hugging Face Datasets

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

#### Chronological

A chronological split is a more realistic evaluation than random splitting for the KEV dataset, as entries are added over time as new vulnerabilities are discovered and exploited. In addition, ransomware tactics and targets may shift over time. One of the weaknesses in the machine learning models from my prior project was that none of them accounted for time.

However, splitting the KEV dataset by the `date_added` column introduces one complication: class imbalance interacts with time. The stratified random split guarantees that both the train and test sets have a representative ratio of ransomware-associated CVEs. A chronological split doesn't guarantee that; if ransomware-flagged CVEs cluster in a particular time window, the test set could end up with very few (or very many) positive examples.

In [8]:
text_df["date_added"] = kev["date_added"]

In [9]:
# Splitting the data into training and testing sets based on the date the vulnerability was added to the KEV dataset

text_df = text_df.sort_values("date_added").reset_index(drop=True)
split_index = int(len(text_df) * 0.8)
train_df_chron = text_df.iloc[:split_index]
test_df_chron = text_df.iloc[split_index:]

In [10]:
train_df_chron = train_df_chron.drop(columns=["date_added"])
test_df_chron = test_df_chron.drop(columns=["date_added"])

In [11]:
# Convert Pandas Dataframes to Hugging Face Datasets

train_dataset_chron = Dataset.from_pandas(train_df_chron)
test_dataset_chron = Dataset.from_pandas(test_df_chron)

## Step 7: Implementing the Models

### Importing Libraries

In [12]:
# For instantiating the models
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

# For evaluating the models
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# For training the models
import transformers
from transformers import DataCollatorWithPadding

# Create a function to set a random seed (for reproducible results)
import random
import torch
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)

### Creating a Method to Evaluate Model Performance

The Confusion Matrix evaluates how well a classification model performs by comparing predicted outcomes against true values.

The upper lefthand corner gives the count of true negatives, the lower lefthand corner gives the count of false negatives, the upper righthand corner gives the count of false positives, and the lower righthand corner gives the count of true positives.

<br>

Accuracy measures how many classifications the algorithm got correct out of every classification it made.

Precision is the ratio of correct positive classifications to all positive classifications made by the model.

Recall is the ratio of correct positive classifications made by the model to all actual positives.

F1-score is a combination of precision and recall. The formula for the f1-score uses a harmonic mean, so it will be low if either precision or recall is low.


In [13]:
def evaluate_model(model, tokenizer, test_dataset, device):
    model.eval()
    all_preds = [] 
    all_labels = []

    with torch.no_grad():
        for row in test_dataset:
            inputs = tokenizer(row["text"], return_tensors="pt", truncation=True, padding=True).to(device)
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()
            
            all_preds.append(pred)
            all_labels.append(row["label"])
    

    cm = confusion_matrix(all_labels, all_preds)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    return cm, accuracy, precision, recall, f1

def print_evaluate_model(cm, accuracy, precision, recall, f1):
    print(f"Confusion matrix: \n{cm}")
    # upper lefthand corner = true negatives
    # lower lefthand corner = false negatives
    # upper righthand corner = false positives
    # lower righthand corner = true positives
    
    print()
    
    print(f"Accuracy score: {accuracy}")
    print(f"Precision score: {precision}")
    print(f"Recall score: {recall}")
    print(f"f1 score: {f1}")

### Loading the Base Model

In [31]:
set_seed() # Ensures reprodcible results from the base model evaluation

device = "cpu"
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model = model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Tokenizing the Data

In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="longest", truncation=True)

#### Standard

In [33]:
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Map:   0%|          | 0/333 [00:00<?, ? examples/s]

#### Chronological

In [34]:
tokenized_train_dataset_chron = train_dataset_chron.map(tokenize_function, batched=True)
tokenized_test_dataset_chron = test_dataset_chron.map(tokenize_function, batched=True)

Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Map:   0%|          | 0/333 [00:00<?, ? examples/s]

### Base Model Evaluation

Since the base model has no exposure to the KEV dataset, poor evaluation scores are anticipated.

In [35]:
print_evaluate_model(*evaluate_model(model, tokenizer, test_dataset, device))

Confusion matrix: 
[[256   7]
 [ 70   0]]

Accuracy score: 0.7687687687687688
Precision score: 0.0
Recall score: 0.0
f1 score: 0.0


### Fine-Tuning the Models

This step exposes the model to the KEV dataset. Although this step is classified as "fine-tuning", this process is akin to training.

In [19]:
def train_model(train_dataset, output_dir, learning_rate):
    set_seed()
    
    # Fresh base model
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model = model.to(device)

    trainer = transformers.Trainer(
        model=model,
        train_dataset=train_dataset,
        args = transformers.TrainingArguments(
            warmup_steps=200,
            logging_steps=10,
            save_steps=200,
            output_dir=output_dir,
            per_device_train_batch_size=12,
            num_train_epochs=3,
            learning_rate=learning_rate,
            optim="adamw_torch"
        ),
        data_collator=DataCollatorWithPadding(tokenizer)
    )

    trainer.train()
    return model

#### Standard

##### Using a learning rate of 1e-4

In [20]:
model_standard_1e4 = train_model(tokenized_train_dataset, "outputs/standard_1e4", 1e-4)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super(

Step,Training Loss
10,0.661291
20,0.605348
30,0.484502
40,0.570958
50,0.545513
60,0.516171
70,0.527176
80,0.487631
90,0.392432
100,0.450230


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

##### Using a learning rate of 4e-5

In [21]:
model_standard_4e5 = train_model(tokenized_train_dataset, "outputs/standard_4e5", 4e-5)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super(

Step,Training Loss
10,0.667312
20,0.644122
30,0.558776
40,0.561750
50,0.539356
60,0.519359
70,0.515162
80,0.491322
90,0.447895
100,0.461572


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

#### Chronological

##### Using a learning rate of 1e-4

In [22]:
model_chron_1e4 = train_model(tokenized_train_dataset_chron, "outputs/chronological_1e4", 1e-4)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super(

Step,Training Loss
10,0.659848
20,0.604661
30,0.466024
40,0.582673
50,0.558520
60,0.540216
70,0.564437
80,0.548657
90,0.488489
100,0.537461


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

##### Using a learning rate of 4e-5

In [23]:
model_chron_4e5 = train_model(tokenized_train_dataset_chron, "outputs/chronological_4e5", 4e-5)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super(

Step,Training Loss
10,0.667120
20,0.644709
30,0.550897
40,0.582204
50,0.564542
60,0.559532
70,0.563744
80,0.563780
90,0.491623
100,0.549861


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\prize\miniconda3\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### Fine-Tuned Model Evaluation

#### Standard

Learning rate: 1e-4

In [25]:
cm_standard_1e4, accuracy_standard_1e4, precision_standard_1e4, recall_standard_1e4, f1_standard_1e4 = evaluate_model(model_standard_1e4, tokenizer, test_dataset, device)
print_evaluate_model(cm_standard_1e4, accuracy_standard_1e4, precision_standard_1e4, recall_standard_1e4, f1_standard_1e4)

Confusion matrix: 
[[239  24]
 [ 46  24]]

Accuracy score: 0.7897897897897898
Precision score: 0.5
Recall score: 0.34285714285714286
f1 score: 0.4067796610169492


##### Learning rate: 4e-5

In [26]:
cm_standard_4e5, accuracy_standard_4e5, precision_standard_4e5, recall_standard_4e5, f1_standard_4e5 = evaluate_model(model_standard_4e5, tokenizer, test_dataset, device)
print_evaluate_model(cm_standard_4e5, accuracy_standard_4e5, precision_standard_4e5, recall_standard_4e5, f1_standard_4e5)

Confusion matrix: 
[[246  17]
 [ 45  25]]

Accuracy score: 0.8138138138138138
Precision score: 0.5952380952380952
Recall score: 0.35714285714285715
f1 score: 0.44642857142857145


#### Chronological

##### Learning rate: 1e-4

In [28]:
cm_chron_1e4, accuracy_chron_1e4, precision_chron_1e4, recall_chron_1e4, f1_chron_1e4 = evaluate_model(model_chron_1e4, tokenizer, test_dataset_chron, device)
print_evaluate_model(cm_chron_1e4, accuracy_chron_1e4, precision_chron_1e4, recall_chron_1e4, f1_chron_1e4)

Confusion matrix: 
[[223  75]
 [ 18  17]]

Accuracy score: 0.7207207207207207
Precision score: 0.18478260869565216
Recall score: 0.4857142857142857
f1 score: 0.2677165354330709


##### Learning rate: 4e-5

In [38]:
cm_chron_4e5, accuracy_chron_4e5, precision_chron_4e5, recall_chron_4e5, f1_chron_4e5 = evaluate_model(model_chron_4e5, tokenizer, test_dataset_chron, device)
print_evaluate_model(cm_chron_4e5, accuracy_chron_4e5, precision_chron_4e5, recall_chron_4e5, f1_chron_4e5)

Confusion matrix: 
[[238  60]
 [ 17  18]]

Accuracy score: 0.7687687687687688
Precision score: 0.23076923076923078
Recall score: 0.5142857142857142
f1 score: 0.3185840707964602


## Conclusion

In the KEV dataset, "Known" values account for only about 20% of the data in the known_ransomware_campaign_use column; 80% is the default accuracy score for a model that defaults to "Unknown". Therefore, a high accuracy score is a misleading indicator of success. Recall score is a better reflection of model performance for my use case. 

The base model classified 0 out of the 70 known instances of ransomware campaign use in the test set. The standard model correctly classified 20 of the 70 instances in the test set, and the chronological model classified 17 of the 35 instances in the chronological test set. These demonstrate that fine-tuning the model was effective at improving the model's performance. However, the improvement is so slight that neither model could be reasonably deployed commercially. 

The plot below shows a comparison between my Logistic Regression model from my machine learning project, my DistilBERT model with a standard stratified split, and my DistilBERT model with a chronological stratified split. Both DistilBERT models shown in the plot had a learning rate of 4e-5, since a slower learning rate produced better results accross all metrics for both models. 

My standard DistilBERT model slightly surpassed my logistic regression model in accuracy and precision, but failed to match my logistic regression model's recall and f1 scores. Interestingly, my chronological DistilBERT model underperformed the other two models across all metrics except recall. This is pertinent because recall is the single most important metric for my use case. However, the improvement in recall score compared to my logistic regression model was insignificant, and the loss in other metrics was too significant to discount. My logistic regression model continues to surpass all other models in terms of well-roundedness. It is my preferred model for identifying ransomware status. 

Although neural networks are broadly considered more advanced than classic machine learning models, they do not always produce better outcomes. Machine learning still has a place in the field of data-based prediction. One limitation of the KEV dataset is that it is relatively small, with only 1665 entires. Neural networks thrive on vast amounts of data, with amount of data directly correlating to model performance. Additionally, the Trainer was run without a held-out validation set during training, so there was no way to monitor for overfitting or underfitting across epochs; future work could incorporate an evaluation split to track this.

Every problem is unique, and experimentation with different models is advisable.

In [36]:
import numpy as np
import matplotlib.pyplot as plt



results = {
    "Logistic Regression":      {"Accuracy": 0.8, 
                                 "Precision": 0.5016611295681063, 
                                 "Recall": 0.45481927710843373,
                                 "F1": 0.47709320695102686},
    "Standard DistilBERT":      {"Accuracy": accuracy_standard_4e5, 
                                 "Precision": precision_standard_4e5, 
                                 "Recall": recall_standard_4e5,
                                 "F1": f1_standard_4e5},
    "Chronological DistilBERT": {"Accuracy": accuracy_chron_4e5, 
                                 "Precision": precision_chron_4e5, 
                                 "Recall": recall_chron_4e5,
                                 "F1": f1_chron_4e5},
}

metrics = ["Accuracy", "Precision", "Recall", "F1"]
models = list(results.keys())

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

for i, model in enumerate(models):
    scores = [results[model][m] for m in metrics]
    ax.bar(x + i * width, scores, width, label=model)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Model Performance Comparison")
ax.legend()

plt.tight_layout()
plt.show()

NameError: name 'f1_chron_4e5' is not defined